# RetainIQ — Retention Action Engine (Rule-Based + ML-Driven Next-Best-Action)

CLAUDE.md Sec 6 documents five churn-driver -> retention-lever pairs (Contract, tenure, Tech support, Payment method, Internet), and CLAUDE.md Sec 14 Phase 4 asks for a "Next-Best-Action engine" on top of the risk tiers (`10-risk-classification.md`) and SHAP/LIME explanations (`11-explainable-ai.md`) already built. Neither of those tells a retention manager *what to do* about an at-risk customer. This notebook demonstrates `src/recommend/action_engine.py`: a ranked action list per customer, combining a tier-appropriate base action (rule-based) with up to two driver-specific actions chosen by the model's own real SHAP output for that customer (ML-driven).

**Scope note:** this is the Next-Best-Action half of CLAUDE.md Sec 14 Phase 4 (the second and final half — `10` shipped the risk-tier half). No optional-LLM insight generation, no FastAPI endpoint, no Streamlit view. See `.claude/specs/13-retention-action-engine.md` for the full spec, including the two worked examples reproduced below.

In [1]:
import sys
from pathlib import Path

# Allow `import src...` when the notebook is run from notebooks/
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from src.data.load_data import clean_data, load_raw_data, TARGET_COLUMN
from src.explain import local_explainer
from src.recommend import action_engine, risk_tiers

pd.set_option("display.max_columns", None)
raw = load_raw_data()
print(f"{len(raw)} raw customers loaded")

7043 raw customers loaded


## 1. The rule tables

`TIER_BASE_ACTIONS` gives each of the four risk tiers one urgency-framed action. `DRIVER_ACTION_RULES` mirrors CLAUDE.md Sec 6's five documented churn signals exactly — no invented lever.

In [2]:
for tier, info in action_engine.TIER_BASE_ACTIONS.items():
    print(f"{tier:>8}: [{info['category']}] {info['action']}")

print()
for rule in action_engine.DRIVER_ACTION_RULES:
    print(f"{rule['feature']:>14} -> [{rule['category']}] {rule['action']}")

Critical: [escalation] Escalate to a retention specialist for a personal outreach call within 24 hours.
    High: [retention_offer] Proactively offer a loyalty discount or service credit.
  Medium: [engagement] Send a targeted engagement email highlighting underused benefits.
     Low: [monitor] No immediate action needed; continue standard engagement monitoring.

      Contract -> [contract] Offer an incentive (discount or loyalty perk) to upgrade from month-to-month to a 1- or 2-year contract.
        tenure -> [onboarding] Enroll the customer in a proactive early-tenure onboarding check-in (first-12-month risk window).
   TechSupport -> [support] Offer a free or discounted Tech Support add-on.
 PaymentMethod -> [payment] Nudge the customer to switch to automatic payment (credit card or bank transfer).
InternetService -> [service_quality] Schedule a proactive fiber-service-quality outreach call.


## 2. Build one shared explainer context

`build_explainer_context` is expensive (~1.9s, per `11-explainable-ai.md`) — built once here and reused for every `recommend_actions_for_customer` call below, exactly the contract `local_explainer.explain_customer` documents.

In [3]:
clean_df = clean_data(raw)
context = local_explainer.build_explainer_context(clean_df)
print("explainer context built")

explainer context built


## 3. Worked example 1 — Critical-tier customer (`5178-LMXOP`)

This customer's real top-3 SHAP drivers (verified during spec research) are all risk-increasing: 1-month tenure, a month-to-month contract, and fiber-optic internet. With `top_n=3`, the tier's escalation action fills slot 1, leaving only 2 driver slots — so the `tenure` and `Contract` rules fire but the `InternetService` rule never gets reached. This is the spec's Requirement 4 slot-cap behavior, not a bug.

In [4]:
critical_customer = raw[raw["customerID"] == "5178-LMXOP"].drop(columns=[TARGET_COLUMN]).to_dict("records")[0]
critical_result = action_engine.recommend_actions_for_customer(critical_customer, explainer_context=context)

print(f"customerID: {critical_result['customerID']}")
print(f"churn_probability: {critical_result['churn_probability']}  (tier: {critical_result['risk_tier']})")
for a in critical_result["actions"]:
    print(f"  [{a['priority']}] ({a['source']}, {a['category']}) {a['action']}")

assert critical_result["risk_tier"] == "Critical"
assert len(critical_result["actions"]) == 3
assert critical_result["actions"][1]["driver_feature"] == "tenure"
assert critical_result["actions"][2]["driver_feature"] == "Contract"
assert not any(a["driver_feature"] == "InternetService" for a in critical_result["actions"])
print("\nMatches the spec's worked example.")

customerID: 5178-LMXOP
churn_probability: 1.0  (tier: Critical)
  [1] (tier, escalation) Escalate to a retention specialist for a personal outreach call within 24 hours.
  [2] (driver, onboarding) Enroll the customer in a proactive early-tenure onboarding check-in (first-12-month risk window).
  [3] (driver, contract) Offer an incentive (discount or loyalty perk) to upgrade from month-to-month to a 1- or 2-year contract.

Matches the spec's worked example.


## 4. Worked example 2 — Low-tier customer (`9763-GRSKD`)

This customer's real top-3 SHAP drivers are one risk-increasing (`Contract="Month-to-month"`) and two protective (`InternetService="DSL"`, `OnlineSecurity="Yes"`, both `direction: "decreases"`). Protective drivers never produce an action (Requirement 4's direction filter), so only 2 of the 3 available slots are filled — not padded with an unrelated generic action.

In [5]:
low_customer = raw[raw["customerID"] == "9763-GRSKD"].drop(columns=[TARGET_COLUMN]).to_dict("records")[0]
low_result = action_engine.recommend_actions_for_customer(low_customer, explainer_context=context)

print(f"customerID: {low_result['customerID']}")
print(f"churn_probability: {low_result['churn_probability']}  (tier: {low_result['risk_tier']})")
for a in low_result["actions"]:
    print(f"  [{a['priority']}] ({a['source']}, {a['category']}) {a['action']}")

assert low_result["risk_tier"] == "Low"
assert len(low_result["actions"]) == 2
assert low_result["actions"][1]["driver_feature"] == "Contract"
print("\nMatches the spec's worked example.")

customerID: 9763-GRSKD
churn_probability: 0.1677  (tier: Low)
  [1] (tier, monitor) No immediate action needed; continue standard engagement monitoring.
  [2] (driver, contract) Offer an incentive (discount or loyalty perk) to upgrade from month-to-month to a 1- or 2-year contract.

Matches the spec's worked example.


## 4b. Expected churn-reduction % (counterfactual re-scoring)

`.claude/specs/15-expected-churn-reduction.md`: for the subset of driver actions whose rule defines a `counterfactual_value` (`Contract`, `TechSupport`, `PaymentMethod`), `recommend_actions_for_customer` re-scores the customer with that one feature flipped, against the same calibrated pipeline used for their own `churn_probability` -- an honest, per-customer, model-derived number, never a static table. `tenure`/onboarding, `InternetService`/service-quality, and the tier-base action have no valid single-feature counterfactual, so they always carry `None` here.

**Note:** this is the model's own single-feature sensitivity for this customer, not a causal treatment-effect estimate -- it answers "what would the model predict for an otherwise-identical customer with this one attribute changed," holding every other feature fixed. Useful for prioritization; not a guaranteed real-world outcome.


In [6]:
for label, result in [("5178-LMXOP (Critical)", critical_result), ("9763-GRSKD (Low)", low_result)]:
    print(label)
    for a in result["actions"]:
        print(f"  [{a['priority']}] ({a['category']}) expected_churn_reduction_pct={a['expected_churn_reduction_pct']}  {a['counterfactual_basis']}")
    print()

critical_contract_action = next(a for a in critical_result["actions"] if a["driver_feature"] == "Contract")
critical_tenure_action = next(a for a in critical_result["actions"] if a["driver_feature"] == "tenure")
critical_tier_action = next(a for a in critical_result["actions"] if a["source"] == "tier")
assert critical_contract_action["expected_churn_reduction_pct"] == 58.6
assert critical_tenure_action["expected_churn_reduction_pct"] is None
assert critical_tier_action["expected_churn_reduction_pct"] is None

low_contract_action = next(a for a in low_result["actions"] if a["driver_feature"] == "Contract")
low_tier_action = next(a for a in low_result["actions"] if a["source"] == "tier")
assert low_contract_action["expected_churn_reduction_pct"] == 14.4
assert low_tier_action["expected_churn_reduction_pct"] is None

print("Matches the spec's Research note exactly.")


5178-LMXOP (Critical)
  [1] (escalation) expected_churn_reduction_pct=None  None
  [2] (onboarding) expected_churn_reduction_pct=None  None
  [3] (contract) expected_churn_reduction_pct=58.6  Contract: 'Month-to-month' -> 'Two year'

9763-GRSKD (Low)
  [1] (monitor) expected_churn_reduction_pct=None  None
  [2] (contract) expected_churn_reduction_pct=14.4  Contract: 'Month-to-month' -> 'Two year'

Matches the spec's Research note exactly.


## 5. A small sample spanning all four tiers

Five more real customers, scored and explained end-to-end, to see the engine's output vary naturally with each customer's own drivers — not just the two hand-picked worked examples above.

In [7]:
sample = raw.sample(n=5, random_state=42).drop(columns=[TARGET_COLUMN])
rows = []
for customer in sample.to_dict("records"):
    result = action_engine.recommend_actions_for_customer(customer, explainer_context=context)
    for a in result["actions"]:
        rows.append({
            "customerID": result["customerID"],
            "risk_tier": result["risk_tier"],
            "priority": a["priority"],
            "category": a["category"],
            "action": a["action"],
        })

sample_actions = pd.DataFrame(rows)
sample_actions

,customerID,risk_tier,priority,category,action
0,1024-GUALD,High,1,retention_offer,Proactively offer a loyalty discount or servic...
1,1024-GUALD,High,2,onboarding,Enroll the customer in a proactive early-tenur...
2,1024-GUALD,High,3,contract,Offer an incentive (discount or loyalty perk) ...
3,0484-JPBRU,Low,1,monitor,No immediate action needed; continue standard ...
4,0484-JPBRU,Low,2,contract,Offer an incentive (discount or loyalty perk) ...
5,3620-EHIMZ,Low,1,monitor,No immediate action needed; continue standard ...
6,6910-HADCM,Critical,1,escalation,Escalate to a retention specialist for a perso...
7,6910-HADCM,Critical,2,onboarding,Enroll the customer in a proactive early-tenur...
8,6910-HADCM,Critical,3,contract,Offer an incentive (discount or loyalty perk) ...
9,8587-XYZSF,Low,1,monitor,No immediate action needed; continue standard ...


## 6. Sanity check: tier vs. action category

Every Critical/High customer in the sample above should get at least one non-`monitor` action if the model attributed any real risk-increasing driver to them — confirming the engine actually differentiates by tier, not just by driver.

In [8]:
cross_tab = sample_actions.groupby("risk_tier", observed=False)["category"].apply(lambda s: sorted(set(s)))
print(cross_tab)

for tier in ["Critical", "High"]:
    if tier in cross_tab.index:
        assert "monitor" not in cross_tab.loc[tier], f"{tier}-tier customer unexpectedly got a 'monitor' action"
print("\nNo Critical/High customer in the sample got a bare 'monitor' action.")

risk_tier
Critical         [contract, escalation, onboarding]
High        [contract, onboarding, retention_offer]
Low                             [contract, monitor]
Name: category, dtype: object

No Critical/High customer in the sample got a bare 'monitor' action.


## Key findings

- **The engine blends rule-based tier framing with ML-driven driver selection** — every action list starts with a tier-appropriate base action, then up to two more actions chosen entirely by which of CLAUDE.md Sec 6's five documented levers the model's own SHAP output flagged as *this specific customer's* real risk drivers.
- **Both hand-verified worked examples reproduce exactly**: the Critical-tier customer (`5178-LMXOP`) gets 3 actions with the `InternetService` rule correctly capped out by `top_n`; the Low-tier customer (`9763-GRSKD`) gets only 2, since its other two top drivers are protective, not risk-increasing.
- **A protective driver never produces an action** — `direction: "decreases"` is filtered before any rule is even checked, so the engine can't recommend "fixing" something that's already working in the customer's favor.
- **`recommend_actions_for_customer` is the function a future Phase 5 `POST /recommend` endpoint will call directly** — no route or dashboard view exists yet; this notebook demonstrates the recommendation contract in isolation, matching the `09`/`10` precedent of shipping a capability before the endpoint that will expose it.